# 34 — Regression × Load Coupling: Intra- vs Inter-Patch Dissociation

**Rank-type tag: `typed_gapfill`. Regime: `[LAB]` (pupil — cannot transfer to WILD).**

NB33 established the intra-patch regression measure (within-AOI leftward
saccades) and its text sensitivity (Gate A; Gate B via height proxy, K4b).
This notebook runs the discriminating test for the **two-mechanism story**:

- *Inter-patch regression* (y-axis, rank-order return) = strategic re-sampling.
  Prior work found it independent of pupil-indexed load (CRForager discriminant:
  ad_penalty ⊥ regression_rate ⊥ mean_lhipa, trial-level rho ≈ 0).
- *Intra-patch regression* (x-axis, within-AOI leftward) = comprehension repair
  (Bicknell & Levy 2010). Repair is triggered by processing difficulty, so it
  **should** couple to load.

**Predicted 2×2:** intra couples to load; inter does not. If both couple (or
neither), the axes are not mechanistically distinct and the CHI 2027 framing
loses its lower rung.

Two grains:
1. **Trial level** — LHIPA (`data_loader.load_lhipa()`; lower = higher load) vs
   per-trial intra-rate and inter-rate. Pooled + within-participant.
2. **Moment level** — landing-fixation pupil diameter (fixation-pupil records,
   index-aligned to fixation-coords with per-trial length guard), z-scored per
   participant: regression landings vs forward landings, per axis.

**Known confound, controlled:** pupil foreshortening — apparent diameter varies
with gaze angle, and regression landings sit systematically left of forward
landings. The moment-level test is re-run after residualizing z-pupil on landing
(x, y) per participant; the claim rests on the residualized version.


In [1]:
import json, glob
from pathlib import Path
from collections import defaultdict
import numpy as np
from scipy.stats import spearmanr, wilcoxon

from data_loader import load_lhipa

ROOT = Path('..').resolve()
FIX_DIR = ROOT / 'AdSERP' / 'data' / 'fixation-coords'
PUP_DIR = ROOT / 'AdSERP' / 'data' / 'fixation-pupil'
AOI_DIR = ROOT / 'data' / 'aoi-typed-gapfill'

THETA = 40
MAX_GAP_MS = 1000
SWEEP_FRAC = 0.5
MIN_VALID_PUPIL = 10     # min valid samples in landing-fixation pupil mean

lhipa = load_lhipa()
print(f'LHIPA trials: {len(lhipa)}')

def load_aois(tid):
    out = []
    for i, a in enumerate(json.load(open(AOI_DIR / f'{tid}.json'))):
        if any(a.get(k) is None for k in ('x', 'y', 'width', 'height')):
            continue
        a['idx'] = i
        out.append(a)
    return out

def assign(x, y, aois):
    best, best_area = None, None
    for a in aois:
        if a['x'] <= x <= a['x'] + a['width'] and a['y'] <= y <= a['y'] + a['height']:
            area = a['width'] * a['height']
            if best is None or area < best_area:
                best, best_area = a, area
    return best


LHIPA trials: 2721


In [2]:
# ── Corpus pass: emit per-event records for both axes ──────────────────
# intra event: within-organic-AOI horizontal-dominant pair, >=THETA, non-sweep
# inter event: consecutive pair spanning two organic AOIs with positions
events = []          # dicts: axis, is_reg, participant, trial, land_x, land_y, pd, pd_ok
trial_counts = defaultdict(lambda: defaultdict(int))
n_pupil_misaligned = 0

for f in sorted(glob.glob(str(AOI_DIR / '*.json'))):
    tid = Path(f).stem
    fp = FIX_DIR / f'{tid}.json'
    if not fp.exists():
        continue
    fixes = json.load(open(fp))
    pup_path = PUP_DIR / f'{tid}.json'
    pup = None
    if pup_path.exists():
        cand = json.load(open(pup_path))
        if len(cand) == len(fixes):
            pup = cand
        else:
            n_pupil_misaligned += 1
    aois = load_aois(tid)
    org = [a for a in aois if a['type'] == 'organic']
    part = tid.split('-')[0]

    seq = [(fx, assign(fx['x'], fx['y'], aois)) for fx in fixes]
    for i in range(1, len(seq)):
        pf, pa = seq[i - 1]
        fx, a = seq[i]
        if pa is None or a is None:
            continue
        if fx['t'] - (pf['t'] + pf['d']) > MAX_GAP_MS:
            continue
        pd_rec = pup[i] if pup else None
        pd_ok = bool(pd_rec and (pd_rec.get('mean_pd') or 0) > 0
                     and (pd_rec.get('n_valid') or 0) >= MIN_VALID_PUPIL)
        base = dict(participant=part, trial=tid,
                    land_x=fx['x'], land_y=fx['y'],
                    pd=pd_rec['mean_pd'] if pd_ok else np.nan, pd_ok=pd_ok)

        if pa['idx'] == a['idx'] and a['type'] == 'organic':
            dx = fx['x'] - pf['x']; dy = fx['y'] - pf['y']
            if abs(dx) >= abs(dy):
                if (dx <= -SWEEP_FRAC * a['width']) and (dy > 0):
                    continue   # return-sweep candidate: neither reg nor fwd
                if dx <= -THETA:
                    events.append({**base, 'axis': 'intra', 'is_reg': 1, 'amp': -dx})
                    trial_counts[tid]['intra_reg'] += 1
                elif dx >= THETA:
                    events.append({**base, 'axis': 'intra', 'is_reg': 0, 'amp': dx})
                    trial_counts[tid]['intra_fwd'] += 1
        elif pa['idx'] != a['idx'] and pa['type'] == 'organic' and a['type'] == 'organic'                 and pa.get('position') is not None and a.get('position') is not None:
            if a['position'] < pa['position']:
                events.append({**base, 'axis': 'inter', 'is_reg': 1})
                trial_counts[tid]['inter_reg'] += 1
            elif a['position'] > pa['position']:
                events.append({**base, 'axis': 'inter', 'is_reg': 0})
                trial_counts[tid]['inter_fwd'] += 1

print(f'events: {len(events)}   trials with events: {len(trial_counts)}')
print(f'pupil-misaligned trials (excluded from moment level): {n_pupil_misaligned}')
for ax in ('intra', 'inter'):
    n_reg = sum(1 for e in events if e['axis'] == ax and e['is_reg'])
    n_fwd = sum(1 for e in events if e['axis'] == ax and not e['is_reg'])
    print(f'  {ax}: reg={n_reg}  fwd={n_fwd}  ({n_reg/(n_reg+n_fwd):.1%} regressive)')


events: 46610   trials with events: 2597
pupil-misaligned trials (excluded from moment level): 0
  intra: reg=11350  fwd=16858  (40.2% regressive)
  inter: reg=7524  fwd=10878  (40.9% regressive)


In [3]:
# ── Grain 1: trial-level LHIPA coupling ────────────────────────────────
# rate = reg / (reg + fwd) per axis per trial; LHIPA lower = higher load.
rows = []
for tid, c in trial_counts.items():
    if tid not in lhipa:
        continue
    r = {'trial': tid, 'participant': tid.split('-')[0], 'lhipa': lhipa[tid]['lhipa']}
    for ax in ('intra', 'inter'):
        denom = c[f'{ax}_reg'] + c[f'{ax}_fwd']
        r[f'{ax}_rate'] = c[f'{ax}_reg'] / denom if denom >= 3 else np.nan
    rows.append(r)

print(f'trials joined with LHIPA: {len(rows)}')
for ax in ('intra', 'inter'):
    x = np.array([r['lhipa'] for r in rows]); y = np.array([r[f'{ax}_rate'] for r in rows])
    ok = ~np.isnan(y)
    rho, p = spearmanr(x[ok], y[ok])
    print(f'pooled  {ax}_rate vs LHIPA: rho={rho:+.3f}  p={p:.1e}  (n={ok.sum()})')

# within-participant: rho per participant (>=20 usable trials), Wilcoxon vs 0
wp = {}
for ax in ('intra', 'inter'):
    per = defaultdict(list)
    for r in rows:
        if not np.isnan(r[f'{ax}_rate']):
            per[r['participant']].append((r['lhipa'], r[f'{ax}_rate']))
    rhos = []
    for p_, v in per.items():
        if len(v) >= 20:
            x, y = zip(*v)
            if len(set(y)) > 1:
                rhos.append(spearmanr(x, y)[0])
    stat, pval = wilcoxon(rhos)
    wp[ax] = (np.mean(rhos), pval, len(rhos))
    print(f'within-participant {ax}: mean rho={np.mean(rhos):+.3f}  '
          f'Wilcoxon-vs-0 p={pval:.1e}  (n={len(rhos)} participants)')


trials joined with LHIPA: 2557
pooled  intra_rate vs LHIPA: rho=-0.031  p=1.5e-01  (n=2158)
pooled  inter_rate vs LHIPA: rho=+0.039  p=8.9e-02  (n=1869)
within-participant intra: mean rho=-0.031  Wilcoxon-vs-0 p=9.1e-02  (n=46 participants)
within-participant inter: mean rho=+0.014  Wilcoxon-vs-0 p=5.9e-01  (n=43 participants)


In [4]:
# ── Grain 2: moment-level pupil at landing, reg vs fwd, per axis ───────
# z-score pupil per participant over all pd_ok events; then per participant,
# mean z(reg) - mean z(fwd) per axis; Wilcoxon across participants.
ev_ok = [e for e in events if e['pd_ok']]
zmap = {}
for p_ in {e['participant'] for e in ev_ok}:
    v = np.array([e['pd'] for e in ev_ok if e['participant'] == p_])
    zmap[p_] = (v.mean(), v.std() if v.std() > 0 else 1.0)
for e in ev_ok:
    m, s = zmap[e['participant']]
    e['z'] = (e['pd'] - m) / s

def moment_test(evs, zkey='z', label=''):
    out = {}
    for ax in ('intra', 'inter'):
        diffs = []
        for p_ in zmap:
            zr = [e[zkey] for e in evs if e['participant'] == p_ and e['axis'] == ax and e['is_reg']]
            zf = [e[zkey] for e in evs if e['participant'] == p_ and e['axis'] == ax and not e['is_reg']]
            if len(zr) >= 10 and len(zf) >= 10:
                diffs.append(np.mean(zr) - np.mean(zf))
        stat, pval = wilcoxon(diffs)
        out[ax] = (np.mean(diffs), pval, len(diffs))
        print(f'{label}{ax}: mean z(reg)-z(fwd) = {np.mean(diffs):+.4f}  '
              f'Wilcoxon p={pval:.1e}  (n={len(diffs)} participants)')
    return out

print(f'moment-level events with valid pupil: {len(ev_ok)}')
raw_res = moment_test(ev_ok, 'z', 'raw        ')


moment-level events with valid pupil: 43271
raw        intra: mean z(reg)-z(fwd) = +0.0149  Wilcoxon p=2.7e-01  (n=46 participants)


raw        inter: mean z(reg)-z(fwd) = +0.0190  Wilcoxon p=8.3e-01  (n=46 participants)


In [5]:
# ── Foreshortening control: residualize z on landing (x, y) ────────────
# Per participant, OLS z ~ 1 + x + y; keep residual. Claim rests on this.
for p_ in zmap:
    idx = [i for i, e in enumerate(ev_ok) if e['participant'] == p_]
    X = np.array([[1.0, ev_ok[i]['land_x'], ev_ok[i]['land_y']] for i in idx])
    z = np.array([ev_ok[i]['z'] for i in idx])
    beta, *_ = np.linalg.lstsq(X, z, rcond=None)
    resid = z - X @ beta
    for j, i in enumerate(idx):
        ev_ok[i]['zres'] = resid[j]

res_res = moment_test(ev_ok, 'zres', 'residualized ')

verdict_intra = 'COUPLES' if res_res['intra'][1] < 0.01 else 'does NOT couple'
verdict_inter = 'COUPLES' if res_res['inter'][1] < 0.01 else 'does NOT couple'
dissoc = (res_res['intra'][1] < 0.01) != (res_res['inter'][1] < 0.01)
print()
print(f'VERDICT (residualized, alpha=0.01): intra {verdict_intra}, inter {verdict_inter}'
      f' -> dissociation {"PRESENT" if dissoc else "ABSENT"}')


residualized intra: mean z(reg)-z(fwd) = -0.0092  Wilcoxon p=9.4e-01  (n=46 participants)
residualized inter: mean z(reg)-z(fwd) = +0.0125  Wilcoxon p=6.3e-01  (n=46 participants)

VERDICT (residualized, alpha=0.01): intra does NOT couple, inter does NOT couple -> dissociation ABSENT


In [6]:
# ── EXPLORATORY: amplitude-stratified rescue of the intra null ─────────
# If the intra pool is diluted (word-grain repairs buried in re-scanning),
# short regressions (<=150 px) should couple where long ones don't.
# Labeled exploratory: specified after seeing the primary null.
strat = {'reg_short': {}, 'reg_long': {}, 'fwd': {}}
for p_ in zmap:
    E = [e for e in ev_ok if e['participant'] == p_ and e['axis'] == 'intra']
    for cls, sel in (('reg_short', lambda e: e['is_reg'] and e.get('amp', 0) <= 150),
                     ('reg_long',  lambda e: e['is_reg'] and e.get('amp', 0) > 150),
                     ('fwd',       lambda e: not e['is_reg'])):
        v = [e['zres'] for e in E if sel(e)]
        if len(v) >= 10:
            strat[cls][p_] = np.mean(v)
for cls in ('reg_short', 'reg_long'):
    common = [p_ for p_ in strat[cls] if p_ in strat['fwd']]
    d = [strat[cls][p_] - strat['fwd'][p_] for p_ in common]
    stat, pval = wilcoxon(d)
    print(f'{cls} vs fwd (residualized): mean diff={np.mean(d):+.4f}  '
          f'Wilcoxon p={pval:.1e}  (n={len(d)})')
print('Rescue FAILS: short-amplitude regressions do not couple either; '
      'the dilution account does not save load coupling at this grain.')


reg_short vs fwd (residualized): mean diff=-0.0049  Wilcoxon p=9.3e-01  (n=46)
reg_long vs fwd (residualized): mean diff=-0.0229  Wilcoxon p=7.6e-01  (n=46)
Rescue FAILS: short-amplitude regressions do not couple either; the dilution account does not save load coupling at this grain.


In [7]:
# ── Key Claims candidates ──────────────────────────────────────────────
print('| ID | Claim | Value |')
print('|---|---|---|')
n_intra = sum(1 for e in events if e['axis'] == 'intra')
n_inter = sum(1 for e in events if e['axis'] == 'inter')
print(f"| K1 | Events (intra / inter, organic) | {n_intra:,} / {n_inter:,} |")
print(f"| K2 | Trial-level within-participant intra_rate x LHIPA | mean rho={wp['intra'][0]:+.3f} (p={wp['intra'][1]:.1e}, n={wp['intra'][2]}) |")
print(f"| K3 | Trial-level within-participant inter_rate x LHIPA | mean rho={wp['inter'][0]:+.3f} (p={wp['inter'][1]:.1e}, n={wp['inter'][2]}) |")
print(f"| K4 | Moment-level intra z(reg)-z(fwd), residualized | {res_res['intra'][0]:+.4f} (p={res_res['intra'][1]:.1e}, n={res_res['intra'][2]}) |")
print(f"| K5 | Moment-level inter z(reg)-z(fwd), residualized | {res_res['inter'][0]:+.4f} (p={res_res['inter'][1]:.1e}, n={res_res['inter'][2]}) |")
print(f"| K6 | Two-mechanism dissociation (alpha=0.01, residualized) | {'PRESENT' if dissoc else 'ABSENT'} |")


| ID | Claim | Value |
|---|---|---|
| K1 | Events (intra / inter, organic) | 28,208 / 18,402 |
| K2 | Trial-level within-participant intra_rate x LHIPA | mean rho=-0.031 (p=9.1e-02, n=46) |
| K3 | Trial-level within-participant inter_rate x LHIPA | mean rho=+0.014 (p=5.9e-01, n=43) |
| K4 | Moment-level intra z(reg)-z(fwd), residualized | -0.0092 (p=9.4e-01, n=46) |
| K5 | Moment-level inter z(reg)-z(fwd), residualized | +0.0125 (p=6.3e-01, n=46) |
| K6 | Two-mechanism dissociation (alpha=0.01, residualized) | ABSENT |


### Caveats

- **Pupil foreshortening** is the load-bearing confound; the raw and residualized
  moment-level results are both reported and only the residualized one is
  claim-grade. Residualization is linear in (x, y); curvature is uncorrected.
- **LHIPA direction:** lower LHIPA = higher load. A *negative* trial-level rho
  for intra_rate means more repair under higher load (prediction); signs must be
  read against this.
- **Pupil lag** (~0.5–1 s) means landing-fixation mean_pd blends pre- and
  post-saccade load; adequate for "load state around the event", not for
  causal ordering of trigger vs response.
- **Inter events here are adjacent-fixation transitions** between organic AOIs
  (gap ≤ 1 s), a stricter and more local definition than NB22's trial-scope
  rank-order returns. The CRForager trial-level independence result is the
  external anchor for the inter axis.
- Return-sweep candidates are excluded from both reg and fwd intra counts.
- **Sensitivity limit:** pupil response lag (~0.5–1 s) is 2–5× fixation duration,
  smearing event-level effects; LHIPA is a trial-scale index. The double null is
  claim-grade for "no coupling detectable at pupil-fixation grain on this
  corpus", not for "repair is effortless".
- **Methods flag for the pupil line (ETTAC):** the raw moment-level intra effect
  (+0.015) reversed sign after residualizing on landing (x, y). Pupil-at-
  regression analyses without gaze-position control will report spurious
  positives at SERP display scale — foreshortening tracks the systematically
  leftward landing positions of regressive saccades.
- `[LAB]`-only; pupil measures cannot earn `[BOTH]`.
